In [1]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('04').getOrCreate()

In [5]:
path = 'C:\\datas\\3rd_week_data\\pga_tour_historical.csv'
pga = spark.read.csv(path,header=True,inferSchema=True)

In [7]:
pga.printSchema()

root
 |-- Player Name: string (nullable = true)
 |-- Season: integer (nullable = true)
 |-- Statistic: string (nullable = true)
 |-- Variable: string (nullable = true)
 |-- Value: string (nullable = true)



In [8]:
pga.show(5)

+---------------+------+----------------+--------------------+-----+
|    Player Name|Season|       Statistic|            Variable|Value|
+---------------+------+----------------+--------------------+-----+
|Robert Garrigus|  2010|Driving Distance|Driving Distance ...|   71|
|   Bubba Watson|  2010|Driving Distance|Driving Distance ...|   77|
| Dustin Johnson|  2010|Driving Distance|Driving Distance ...|   83|
|Brett Wetterich|  2010|Driving Distance|Driving Distance ...|   54|
|    J.B. Holmes|  2010|Driving Distance|Driving Distance ...|  100|
+---------------+------+----------------+--------------------+-----+
only showing top 5 rows



## 1. `describe`와 `summary`

- `DataFrame.describe('col1','col2')`: 지정된 column에 대한 통계값(`count,mean,stddev,min,max`)을 갖는 `DataFrame`반환
>- `stddev`는 표준편차임
- `DataFrame.describe('통계값1','통계값2')`: `DataFrame` 각 column에 대해 지정된 통계값을 갖는 `DataFrame` 반환 

In [10]:
pga.describe('Season').show()

+-------+------------------+
|summary|            Season|
+-------+------------------+
|  count|           2740403|
|   mean| 2013.973479083186|
| stddev|2.6070501155207104|
|    min|              2010|
|    max|              2018|
+-------+------------------+



In [14]:
# 열의 값이 str이라도, 오류는 발생하지 않으나, NULL값으로 반환됨
pga.summary('mean','count').select('Season').show()

+-----------------+
|           Season|
+-----------------+
|2013.973479083186|
|          2740403|
+-----------------+



## 2. **parquet** file 쓰기

- **parquet** 파일은 하둡에서 사용하는 **column** 기반의 데이터 (cf> `DataFrame`은 **Row**기반의 데이터구조)
- 효율적으로 데이터를 저장할 수 있음 (csv 대비 75% 이상 용량을 줄일 수 있음)
- **pqrquet** 파일의 메타데이터에 일부 통계정보가 포함되어 있어, 속도 향상에 효과가 있음
- 단, 데이터 추가, 삭제 등의 작업은 까다로움

In [17]:
pga.write.mode('overwrite').parquet('C:\\datas\\3rd_week_data\\pga_parquet\\')

- 여러 개의 parquet -> 여러 개의 executor에서 작업 후 반환

## 3. **parquet** file 읽기

In [19]:
pga_parquet = spark.read.parquet('C:\\datas\\3rd_week_data\\pga_parquet\\')
pga_parquet.show(5)

+---------------+------+--------------------+--------------------+-----+
|    Player Name|Season|           Statistic|            Variable|Value|
+---------------+------+--------------------+--------------------+-----+
| Scott McCarron|  2011|Birdie or Better ...|Birdie or Better ...|   14|
|Carl Pettersson|  2011|Birdie or Better ...|Birdie or Better ...|   22|
| Brendan Steele|  2011|Birdie or Better ...|Birdie or Better ...|   15|
|    Arjun Atwal|  2011|Birdie or Better ...|Birdie or Better ...|   27|
|Graeme McDowell|  2011|Birdie or Better ...|Birdie or Better ...|    7|
+---------------+------+--------------------+--------------------+-----+
only showing top 5 rows



## 4. **partitioned parquet** file 쓰기

- `write`의 `partitionBy`는 지정된 column을 기준으로 분리하여 `parquet` 파일을 저장할 수 있게 함

In [20]:
pga.write.mode('overwrite').partitionBy('Season').parquet('C:\\datas\\3rd_week_data\\season_pga')

## 5. **partitioned parquet** file 읽기

In [21]:
season_pga = spark.read.parquet('C:\\datas\\3rd_week_data\\season_pga')
season_pga.show(5)

+----------------+--------------------+--------------------+-----+------+
|     Player Name|           Statistic|            Variable|Value|Season|
+----------------+--------------------+--------------------+-----+------+
|   Blayne Barber|GIR Percentage - ...|GIR Percentage - ...|63.96|  2016|
|   Rob Oppenheim|Approaches from i...|Approaches from i...|   67|  2016|
|     Vijay Singh|GIR Percentage - ...|GIR Percentage - ...|63.64|  2016|
|   Abraham Ancer|Approaches from i...|Approaches from i...|   51|  2016|
|Henrik Norlander|GIR Percentage - ...|GIR Percentage - ...|63.59|  2016|
+----------------+--------------------+--------------------+-----+------+
only showing top 5 rows



- 특정 **partitioned parquet** file 선택하여 읽기

In [22]:
spark.read.parquet('C:\\datas\\3rd_week_data\\season_pga\\Season=2012',
                  'C:\\datas\\3rd_week_data\\season_pga\\Season=2014',
                  'C:\\datas\\3rd_week_data\\season_pga\\Season=2017').show(5)

+---------------+--------------------+--------------------+-----+
|    Player Name|           Statistic|            Variable|Value|
+---------------+--------------------+--------------------+-----+
| Johnson Wagner|First Tee Early P...|First Tee Early P...|   98|
|Scott Stallings| Putting from 15-25'|Putting from 15-2...|   35|
|    Blake Adams|First Tee Early P...|First Tee Early P...|  103|
| Rory Sabbatini| Putting from 15-25'|Putting from 15-2...|   36|
|  Jonathan Byrd|First Tee Early P...|First Tee Early P...|   67|
+---------------+--------------------+--------------------+-----+
only showing top 5 rows

